In [ ]:
#FX for the training dataset
import torch
import torchvision.transforms as T

center_crop = T.CenterCrop(size=224)  # scegli la size del crop

def make_ssl_batch(x):
    """
    x: immagini di input (B, C, H, W)
    Ritorna:
        x_ssl: immagini croppate+ruotate (B, C, h, w)
        y_ssl: label rotazione (B,) in {0,1,2,3}
    """
    B, C, H, W = x.shape
    x_ssl_list = []
    y_ssl_list = []

    # possibili rotazioni in multipli di 90°
    angles = [0, 90, 180, 270]
    ks = [0, 1, 2, 3]  # corrispondente k per torch.rot90

    for i in range(B):
        img = x[i]                              # (C,H,W)
        img = center_crop(img)                  # center crop

        # scegli random una delle 4 rotazioni
        idx = torch.randint(0, 4, (1,)).item()
        k = ks[idx]

        # rotazione di k*90° senza interpolazione
        img_rot = torch.rot90(img, k=k, dims=(1, 2))

        x_ssl_list.append(img_rot)
        y_ssl_list.append(idx)

    x_ssl = torch.stack(x_ssl_list, dim=0)          # (B,C,h,w)
    y_ssl = torch.tensor(y_ssl_list, dtype=torch.long, device=x.device)  # (B,)

    return x_ssl, y_ssl


#ssl_head, da usare nel branch SSL
class SSLHead(nn.Module):
    def __init__(self, in_channels, num_classes=4):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 32, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, feat):
        x = F.relu(self.conv(feat))
        x = self.pool(x).flatten(1)  # (B,32)
        return self.fc(x)            # (B,4)


In [ ]:
#MAIN
if __name__ == '__main__':
  seed=42
  device="cuda" if torch.cuda.is_available() else "cpu"
  DATA_PATH="todo"
  MODEL_SAVE_PATH="todo"
  torch.manual_seed(seed)
  np.random.seed(seed)

  model=Unet(n_classes=1)
  ssl_head=SSLHead(in_channels=224)
  model=model.to(device)

  model.apply(init_weights)

  #apply datasplit and train dataloader TODO

  batch_size=8 #try with 8, then 12 and 16
  epochs_per_round=6
  warmup_rounds=1
  total_rounds=40
  gamma=0.9
  optimizer=torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
  criterion=L_supBCEDice()

  for round in range(total_rounds):
    #warm start TODO
    for epoch in range(epochs_per_round):
      model.train()
      train_running_loss=0
      for img, img_mask, is_pos, rot_img, rot_label in loader:
        img,mask=img.to(device),img_mask.to(device)
        is_pos=is_pos.to(device)
        optimizer.zero_grad()

        #segmentation path and loss
        logits=model(img)
        loss_bce_dice=criterion(y_pred,mask,is_pos)

        #ssl path and loss
        x_ssl, y_ssl = make_ssl_batch(img)
        _,ssl_logits=ssl_head(x_ssl)
        L_ssl=F.cross_entropy(ssl_logits,y_ssl)



        #total loss
        L_tot=gamma*loss_bce_dice+(1-gamma)*L_ssl
        train_running_loss+=L_tot.item()

        L_tot.backward()
        optimizer.step()

      train_loss=train_running_loss/len(loader)
      print(f"Train loss: {train_loss}")
